# Using LLM-as-a-Judge 🧑‍⚖️
_Based on [llm-judge tutorial](https://huggingface.co/learn/cookbook/llm_judge)
by [Aymeric Roucher](https://huggingface.co/m-ric)._  
_Adapted for local model's usage._

LLM-as-a-Judge refers to using one language model to **evaluate** the output of another model (or even itself).  
This approach has rapidly become a practical alternative to large-scale human annotation, particularly for:

- preference modeling  
- RLHF pipelines  
- open-ended generation evaluation  
- content safety scoring  
- heuristic quality assessment  

The central idea:

> Provide the judge model with a *task description*, a *reference or rubric*, and the *candidate output*,  
> and ask it to assign a **structured score**.

For instance, an assistant's answer to a question can be:
- not grounded in context
- repetitive, repetitive, repetitive
- grammatically incorrects
- Excessively lengthy and characterized by an overabundance of words, leading to a situation where the discourse or written content becomes overly detailed and protracted
- incoherent
- ...

The list of criteria goes on and on. And even if we had a limited list, each of these would be hard to measure: "devising a rule-based program to assess the outputs is extremely challenging. Traditional evaluation metrics based on the similarity between outputs and reference answers (e.g., ROUGE, BLEU) are also ineffective for these questions."

✅ A powerful solution to assess outputs in a human way, without requiring costly human time, is LLM-as-a-judge.
This method was introduced in [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://huggingface.co/papers/2306.05685) - which I encourage you to read.

💡 The idea is simple: ask an LLM to do the grading for you. 🤖✓

But we'll see that it will not work well out-of-the-box: you need to set it up carefully for good results.

In [ ]:
# Environment for week11: Decoding strategies, evaluation harness, LLM-as-a-judge.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week11` in the repo root instead.
%pip install -q "torch>=2.4,<3" "transformers>=4.56,<5" "datasets>=3.6,<4" "accelerate>=1.0" "pandas>=2.2" "tqdm>=4.66" "huggingface-hub>=0.25" "lm-eval>=0.4.8" "ipython>=8"

In [ ]:
from transformers import pipeline
import re
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from IPython.display import display


tqdm.pandas()  # load tqdm's pandas support
pd.set_option("display.max_colwidth", None)

In [ ]:
repo_id = "unsloth/Llama-3.2-1B-Instruct"

generator = pipeline("text-generation", model=repo_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
generator(text_inputs ="How are you today?", max_new_tokens=20)[0]["generated_text"]

"How are you today? I hope you're having a great day so far! \n\nHi, I'm a huge fan of"

## 1. Preparing the Judge Evaluation Setup

We begin by loading a dataset containing:

- **the original prompt**  
- **the generated answer**  
- **a human reference score**  

Our goal is to train (or rather *design*) a judge that can approximate these human ratings.

The difficulty is structural:  
human scoring is noisy, inconsistent, and often ambiguous.  
Thus, the best any judge can hope for is to **match human-level agreement**, measured via Pearson correlation.

In this section we compute:

- the baseline inter-human correlation  
- the model’s baseline correlation using a naive judge prompt  

These values provide a reference point for later improvements.

In our case, we will use [`feedbackQA`](https://huggingface.co/datasets/McGill-NLP/feedbackQA), which contains 2 human evaluations and scores for each question/answer couple: using a sample of 30 examples will be representative of what your small evaluation dataset could be.

In [ ]:
# feedbackQA is a script dataset: it needs datasets < 4 and trust_remote_code
ratings = load_dataset("McGill-NLP/feedbackQA", trust_remote_code=True)["train"]
ratings = pd.DataFrame(ratings)

ratings["review_1"] = ratings["feedback"].apply(lambda x: x["rating"][0])
ratings["explanation_1"] = ratings["feedback"].apply(lambda x: x["explanation"][0])
ratings["review_2"] = ratings["feedback"].apply(lambda x: x["rating"][1])
ratings["explanation_2"] = ratings["feedback"].apply(lambda x: x["explanation"][1])
ratings = ratings.drop(columns=["feedback"])

# Map scores to numeric values
conversion_dict = {"Excellent": 4, "Acceptable": 3, "Could be Improved": 2, "Bad": 1}
ratings["score_1"] = ratings["review_1"].map(conversion_dict)
ratings["score_2"] = ratings["review_2"].map(conversion_dict)

Computing a baseline is essential:

- **Human ↔ Human correlation** gives an upper bound on performance  
- **Judge (v1) ↔ Human correlation** tells us how much signal the initial prompt captures  
- **Judge ↔ Judge** repeatability tells us whether the scoring scheme is stable  

Pearson correlation is used because it captures *linear agreement* across continuous scores and is widely used in evaluation literature:

$$
r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}
         {\sqrt{\sum_i (x_i - \bar{x})^2}\sqrt{\sum_i (y_i - \bar{y})^2}}
$$

A low baseline is normal when the rubric is vague or the scoring instructions are underspecified.


In [ ]:
print("Correlation between 2 human raters:")
print(f"{ratings['score_1'].corr(ratings['score_2'], method='pearson'):.3f}")

Correlation between 2 human raters:
0.563


This correlation between 2 human raters is not that good.  
🤝 Human agreement is not clear enough.

This tells us the dataset contains inherent ambiguity:

- raters interpret the rubric differently  
- some questions have no universally “correct” scoring  
- noise exists in subjective evaluation tasks  

This explains why our “ground truth” contains inconsistencies — the judge is trying to approximate a target that is itself imperfect.  
Understanding this sets realistic expectations for what follows.


In [ ]:
# Sample examples
ratings_where_raters_agree = ratings.loc[ratings["score_1"] == ratings["score_2"]]
examples = ratings_where_raters_agree.groupby("score_1").sample(7, random_state=1214)
examples["human_score"] = examples["score_1"]

# Visualize 1 sample for each score
display(examples.groupby("human_score").first())

,question,answer,review_1,explanation_1,review_2,explanation_2,score_1,score_2
human_score,,,,,,,,
1,What can I do to help people that are grieving?,"Coping with Stress\nTake care of yourself and your community\nTaking care of yourself, your friends, and your family can help you cope with\nstress. Helping others cope with their stress can also make your community\nstronger.\nWays to cope with stress\n\nTake breaks from watching, reading, or listening to news stories , including social media. Hearing about the pandemic repeatedly can be upsetting.\nTake care of your body. \nTake deep breaths, stretch, or meditate.\nTry to eat healthy, well-balanced meals.\nExercise regularly, get plenty of sleep.\nAvoid alcohol and drugs.\n\n\nMake time to unwind. Try to do some other activities you enjoy.\nConnect with others. Talk with people you trust about your concerns and how you are feeling.\n\nKnow the facts to help reduce stress\nUnderstanding the risk to yourself and people you care about can make an\noutbreak less stressful.\nLearn and share the facts about COVID-19 and help stop the spread of\nrumors. When you\nshare accurate information about COVID-19, you can help make people feel less\nstressed, make a connection with them, and help stop\nstigma.\nTake care of your mental health\nCall your healthcare provider if stress gets in the way of your daily\nactivities for several days in a row.\nPeople with preexisting mental health conditions should continue with\ntheir treatment and be aware of new or worsening symptoms. Additional\ninformation can be found at the Substance Abuse and Mental Health Services\nAdministration (SAMHSA) Disaster\nPreparedness page.\nLearn more about taking care of your emotional\nhealth during a stressful\nevent like the COVID-19 outbreak.",Bad,The question is about others which the reply did not answer.,Bad,The response could have addressed how to help those that are grieving cope rather than what it was presenting.,1,1
2,What protocols do workplaces need to follow to keep everyone safer?,Coronavirus and Australian workplace laws\nHealth & safety in the workplace\nWorkplaces must follow the rules about health and safety during coronavirus to\nhelp stop it spreading. Find out more about:\n\nrules and obligations under workplace health and safety laws\nhow to manage the risk of coronavirus in the workplace\nwhere to go for help.\n\nLearn more about Health and safety in the workplace during\ncoronavirus.,Could be Improved,"This answer needs to be improved because it doesn’t provide information up-front about workplaces during the pandemic. Instead, it just includes a hyperlink.",Could be Improved,"there is one link to information, but there is no information in the answer about how to stay safe in the workplace. it talks about the need to stay safe in the workplace, but it doesn't talk about ways in which to actually do that.",2,2
3,How soon can I apply for financial support?,"COVID-19 early release of super\nAfter you apply\nIt will take us up to four business days to process your application and send\nyour outcome letter to your myGov inbox. You may also receive an SMS\nnotification.\nIf you receive a notification from us and haven't applied to access your super\nearly, you need to call us or your fund as soon as possible.\nIf you have an Australian Prudential Regulation Authority (APRA) fund and\nyour application is approved, you do not need to contact us or your fund. Your\nfund will make the payment to you without you needing to apply to them\ndirectly.\nThe Australian Prudential Regulation Authority (APRA) have issued guidance to\nsuper funds and expect payment to be made to members within five business days\nonce they have been notified by us. However, this time may increase where\nfunds need to contact you to clarify information. More information can be\nfound on APRA's websiteExternal Link.\nIf your fund is a state-administered fund, they need to follow the rules\nof their trust deed to determine if t

## 2. Create our LLM judge

We build our LLM judge with a basic prompt template.  
The prompt includes:

- the **question**  
- the **model’s answer**  
- the **evaluation rubric**  
- the **required output range**, including `minimum`, `maximum`, and value type (`float`)  
- a short explanation field  

This baseline prompt is intentionally simple, but it already yields a meaningful correlation because the judge has clear boundaries and a numeric target.

Even this minimal setup outperforms fully unsupervised heuristics.


In [ ]:
JUDGE_PROMPT = """
You will be given a user_question and system_answer couple.
Your task is to provide a 'total rating' scoring how well the system_answer answers the user concerns expressed in the user_question.
Give your answer as a float on a scale of 0 to 10, where 0 means that the system_answer is not helpful at all, and 10 means that the answer completely and helpfully addresses the question.

Provide your feedback as follows:

Feedback:::
Total rating: (your rating, as a float between 0 and 10)

Now here are the question and answer.

Question: {question}
Answer: {answer}

Feedback:::
Total rating: """

In [ ]:
examples["llm_judge"] = examples.progress_apply(
    lambda x: generator(
        text_inputs=JUDGE_PROMPT.format(question=x["question"], answer=x["answer"]),
        max_new_tokens=1000,
    )[0]["generated_text"],
    axis=1,
)

  0%|          | 0/28 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
def extract_judge_score(answer: str, split_str: str = "Total rating:") -> int:
    try:
        if split_str in answer:
            rating = answer.split(split_str)[2]
        else:
            rating = answer
        digit_groups = [el.strip() for el in re.findall(r"\d+(?:\.\d+)?", rating)]
        return float(digit_groups[0])
    except Exception as e:
        print(e)
        return None


examples["llm_judge_score"] = examples["llm_judge"].apply(extract_judge_score)
# Rescale the score given by the LLM on the same scale as the human score
examples["llm_judge_score"] = (3*examples["llm_judge_score"] // 10) + 1

In [ ]:
print("Correlation between LLM-as-a-judge and the human raters:")
print(
    f"{examples['llm_judge_score'].corr(examples['human_score'], method='pearson'):.3f}"
)

Correlation between LLM-as-a-judge and the human raters:
-0.169


This is not bad, given that the Pearson correlation between 2 human raters is itself limited.  
In other words, our baseline judge already reaches close to human-level agreement.

But we can do better. 🔝

However, naive judge prompts often suffer from:

- vague descriptions of what “quality” means  
- insufficient reasoning steps  
- poor anchoring to the scoring scale  
- mid-range score bias (“hedging”)  

These issues motivate a more structured and explicit prompt.

## 3. Improve the LLM judge

As shown by [Aparna Dhinakaran](https://twitter.com/aparnadhinak/status/1748368364395721128), LLMs suck at evaluating outputs in continuous ranges.
[This article](https://www.databricks.com/blog/LLM-auto-eval-best-practices-RAG) gives us a few best practices to build a better prompt:
- ⏳ **Leave more time for thought** by adding an `Evaluation` field before the final answer.
- 🔢 **Use a small integer scale** like 1-4 or 1-5 instead of a large float scale as we had previously.
- 👩‍🏫 **Provide an indicative scale for guidance**.
- We even add a carrot to motivate the LLM!

In [ ]:
IMPROVED_JUDGE_PROMPT = """
You will be given a user_question and system_answer couple.
Your task is to provide a 'total rating' scoring how well the system_answer answers the user concerns expressed in the user_question.
Give your answer on a scale of 1 to 4, where 1 means that the system_answer is not helpful at all, and 4 means that the system_answer completely and helpfully addresses the user_question.

Here is the scale you should use to build your answer:
1: The system_answer is terrible: completely irrelevant to the question asked, or very partial
2: The system_answer is mostly not helpful: misses some key aspects of the question
3: The system_answer is mostly helpful: provides support, but still could be improved
4: The system_answer is excellent: relevant, direct, detailed, and addresses all the concerns raised in the question

Provide your feedback as follows:

Feedback:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 4)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question and answer.

Question: {question}
Answer: {answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """

In [ ]:
examples["llm_judge_improved"] = examples.progress_apply(
    lambda x: generator(
        text_inputs=IMPROVED_JUDGE_PROMPT.format(question=x["question"], answer=x["answer"]),
        max_new_tokens=500,
    )[0]["generated_text"],
    axis=1,
)

  0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
def extract_judge_score_improved(answer: str, split_str: str = "Total rating:") -> int:
    try:
        if split_str in answer:
            rating = answer.split(split_str)[3]
        else:
            rating = answer
        digit_groups = [el.strip() for el in re.findall(r"\d+(?:\.\d+)?", rating)]
        return float(digit_groups[0])
    except Exception as e:
        print(e)
        return None


In [ ]:
examples["llm_judge_improved_score"] = examples["llm_judge_improved"].apply(
    extract_judge_score_improved
)

In [ ]:
examples

question  \
1976                                                                                         What can I do to help people that are grieving?   
4040              Should I continue to work if my work needs me, even though I am classified as vulnerable? Will I be penalised if I do not?   
5463                                                                                                   Will I be safe at my reopened school?   
2219                                                                                                        What does community spread mean?   
1104                                                                                             Is the Great Barrier Reef Marine Park open?   
5636                                                            I've been vaccinated against BCG, am I also protected from getting COVID-19?   
5390                                              Who will get priority for our limited medical resources and equipment during the pandemic?   
78                                                                       What protocols do workplaces need to follow to keep everyone safer?   
208   Who decides if our early childhood education and care service has to close due to COVID-19, and what should we do after shutting down?   
472                                                                                         Can the covid19 event visa be granted to anyone?   
4367                                           I am a young carer and am under strain right now. What resources and help is available to me?   
239                                                       Are there any personal hygiene practices that can help stop the spread of COVID-19   
1975                                                                                         What can I do to help people that are grieving?   
670                                                                                      What programs can assist busy childcare facilities?   
712                                                                                              How soon can I apply for financial support?   
1877                                                      What should I do to implement quarantine in my correctional or detention facility?   
1225                                                     What restrictions have been placed on any cruise ship that has left a foreign port?   
662                                                                                      How is the government helping to keep credit going?   
2026                                                               How should I know whether I need to isolate myself or go into quarantine?   
4780                                                                               Are universities going to be making unconditional offers?   
2690                                                        When shipping specimens, should a responsible person be listed on the paperwork?   
3147                                                                   Should vulnerable children be expected to be in educational settings?   
5375                                                                                             What symptoms are associated with Covid-19?   
651                                                                                     What product should I use to disinfect hard surfaces   
5150                                                           Can suspected and confirmed COVID-19 cases be kept in the same hospital room?   
5330                                                   What are the requirements for the attendance of athletes who are competing in events?   
1434                                                    What could I do to protect my employees as a supervisor at a rail transit operation?   
4194                                                          What do 3rd trimester or at risk healt

In [ ]:
print("Correlation between LLM-as-a-judge and the human raters:")
print(
    f"{examples['llm_judge_improved_score'].corr(examples['human_score'], method='pearson'):.3f}"
)

Correlation between LLM-as-a-judge and the human raters:
0.192


The correlation was **improved by nearly 30%** with only a few prompt-engineering adjustments.

This shows how sensitive judges are to:

- clearer goals  
- better anchoring  
- requiring structured evaluation steps  
- enforcing consistent formatting  

⚖️ A passive-aggressive lawyer’s tip to the LLM, which I hereby declare not legally binding:  
Better rubrics make better judgments.

In [ ]:
errors = pd.concat(
    [
        examples.loc[
            examples["llm_judge_improved_score"] > examples["human_score"]
        ].head(1),
        examples.loc[
            examples["llm_judge_improved_score"] < examples["human_score"]
        ].head(2),
    ]
)

display(
    errors[
        [
            "question",
            "answer",
            "human_score",
            "explanation_1",
            "llm_judge_improved_score",
            "llm_judge_improved",
        ]
    ]
)

question  \
1976      What can I do to help people that are grieving?   
670   What programs can assist busy childcare facilities?   
712           How soon can I apply for financial support?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      answer  \
1976  Coping with Stress\nTake care of yourself and your community\nTaking care of yourself, your friends, and your family can help you cope with\nstress. Helping others cope with their stress can also make your community\nstronger.\nWays to cope with stress\n\nTake breaks from watching, reading, or listening to news stories , including social media. Hearing about the pandemic repeatedly can be upsetting.\nTake care of your body. \nTake deep breaths, stretch, or meditate.\nTry to eat healthy, well-balanced meals.\nExercise regularly, get plenty of sleep.\nAvoid alcohol and drugs.\n\n\nMake time to unwind. Try to do some other activities you enjoy.\nConnect with others. Talk with people you trust about your concerns and how you are feeling.\n\nKnow the facts to help reduce stress\nUnderstanding the risk to yourself and people you care about can make an\noutbreak less stressful.\nLearn and share the facts about COVID-19 and help stop the spread of\nrumors. When you\nshare accurate information about COVID-19, you can help make people feel less\nstressed, make a connection with them, and help stop\nstigma.\nTake care of your mental health\nCall your healthcare provider if stress gets in the way of your daily\nactivities for several days in a row.\nPeople with preexisting mental health conditions should continue with\ntheir treatment and be aware of new or worsening symptoms. Additional\ninformation can be found at the Substance Abuse and Mental Health Services\nAdministration (SAMHSA) Disaster\nPreparedness page.\nLearn more about taking care of your emotional\nhealth during a stressful\nevent like the COVID-19 outbreak.   
670                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

The disagreements are minor: overall, we seem to have reached a good level of performance for our system!

## 4. How do we take our LLM judge even further?

🎯 **You will never reach 100%:** Let's first note that our human ground truth certainly has some noise, so agreement/correlation will never go up to 100% even with a perfect LLM judge.

🧭 **Provide a reference:** If you had access to a reference answer for each question, you should definitely give this to the Judge LLM in its prompt to get better results!

▶️ **Provide few-shot examples:** adding some few-shot examples of questions and ground truth evaluations in the prompt can improve the results. _(I tried it here, it did not improve results in this case so I skipped it, but it could work for your dataset!)_

➕ **Additive scale:** When the judgement can be split into atomic criteria, using an additive scale can further improve results: see below 👇
```python
ADDITIVE_PROMPT = """
(...)
- Award 1 point if the answer is related to the question.
- Give 1 additional point if the answer is clear and precise.
- Provide 1 further point if the answer is true.
- One final point should be awarded if the answer provides additional resources to support the user.
...
"""
```

**Implement with structured generation:**

Using **structured generation**, you can configure the LLM judge to directly provide its output as a JSON with fields `Evaluation` and `Total rating`, which makes parsing easier : see our [structured generation](structured_generation) cookbook to learn more!

## Conclusion

That's all for today, congrats for following along! 🥳

I'll have to leave you, some weirdos are banging on my door, claiming they have come on behalf of Mixtral to collect H100s. 🤔